In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.cluster import KMeans
CSV_PATH = 'df_crashes.csv'
OUTPUT_FILE = 'severity_hotspots.html'
CLUSTER_OPTIONS = [3, 5, 10, 20, 30]
DEFAULT_CLUSTERS = 10
START_DATE = '2018-01-01'

LAT_COL = 'LATITUDE'
LON_COL = 'LONGITUDE'
HOUR_COL = 'CRASH_HOUR'
DATE_COL = 'CRASH_DATE'
df = pd.read_csv(CSV_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
df = df.dropna(subset=[LAT_COL, LON_COL, HOUR_COL, DATE_COL])
df[HOUR_COL] = df[HOUR_COL].astype(int)
df = df[(df[HOUR_COL] >= 0) & (df[HOUR_COL] <= 23)]
if START_DATE:
    df = df[df[DATE_COL] >= pd.to_datetime(START_DATE)]
df['INJURIES_TOTAL'] = pd.to_numeric(df['INJURIES_TOTAL'], errors='coerce').fillna(0)
df['INJURIES_FATAL'] = pd.to_numeric(df['INJURIES_FATAL'], errors='coerce').fillna(0)
df['INJURIES_INCAPACITATING'] = pd.to_numeric(df['INJURIES_INCAPACITATING'], errors='coerce').fillna(0)
df['INJURIES_SEVERE'] = df['INJURIES_FATAL'] + df['INJURIES_INCAPACITATING']
df['HAS_INJURY'] = df['INJURIES_TOTAL'] > 0
df['HAS_SEVERE'] = df['INJURIES_SEVERE'] > 0

center_lat = df[LAT_COL].mean()
center_lon = df[LON_COL].mean()
hour_labels = [f'{h}:00' for h in range(24)]

print(f"Total Crashes: {len(df):,}")
print(f"Crashes with Any Injury: {len(df[df['HAS_INJURY']]):,}")
print(f"Crashes with Severe Injury: {len(df[df['HAS_SEVERE']]):,}")

# ============================================
# GENERATION LOOP
# ============================================
data_store = {}
max_val_map = {}

print("Generating Severity Hotspots...", CLUSTER_OPTIONS)

for k in CLUSTER_OPTIONS:
    data_store[k] = {}
    local_max_val = 0

    for hour in range(24):
        hdf = df[df[HOUR_COL] == hour]
        data_store[k][hour] = {}

        scenarios = [
            ('Severe', hdf[hdf['HAS_SEVERE'] == True], 'INJURIES_SEVERE'),
            ('Total', hdf[hdf['HAS_INJURY'] == True], 'INJURIES_TOTAL')
        ]

        for name, subset, weight_col in scenarios:
            n_points = len(subset)
            if n_points == 0:
                data_store[k][hour][name] = dict(
                    lat=[], lon=[], counts=[], metric=[]
                )
                continue

            # K-Means
            n_clusters = min(k, n_points)
            coords = subset[[LAT_COL, LON_COL]].to_numpy()
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=3)
            labels = kmeans.fit_predict(coords)
            centers = kmeans.cluster_centers_

            # Aggregate
            counts = np.bincount(labels, minlength=n_clusters)
            metric_sum = np.zeros(n_clusters)
            w_vals = subset[weight_col].to_numpy()

            for idx, lab in enumerate(labels):
                metric_sum[lab] += w_vals[idx]

            if metric_sum.max() > local_max_val:
                local_max_val = metric_sum.max()

            data_store[k][hour][name] = dict(
                lat=centers[:, 0],
                lon=centers[:, 1],
                counts=counts,
                metric=metric_sum
            )

    max_val_map[k] = local_max_val if local_max_val > 0 else 1

# ============================================
# TRACE BUILDER
# ============================================
def make_trace(data_dict, name, color_scale, max_val, visible=None):
    lat = np.array(data_dict['lat'])
    lon = np.array(data_dict['lon'])

    common_args = dict(name=name)
    if visible is not None:
        common_args['visible'] = visible

    if len(lat) == 0:
        return go.Scattermapbox(lat=[], lon=[], mode='markers', marker=dict(size=0), **common_args)

    counts = np.array(data_dict['counts'])
    metric = np.array(data_dict['metric'])

    # Sizing
    sizes = 12 + 20 * (counts / (counts.max() + 1))

    # Color Normalization (0 to 1 relative to this K's max)
    colors = metric / max_val

    hover = [
        f"<b>{name} Hotspot</b><br>Crashes: {c:,}<br>Injuries: {m:,.0f}"
        for c, m in zip(counts, metric)
    ]



    return go.Scattermapbox(
        lat=lat, lon=lon, mode='markers',
        marker=dict(
            size=sizes,
            color=colors,
            colorscale=color_scale,
            cmin=-0.1,
            cmax=0.8,
            opacity=0.95,
            showscale=False
        ),
        text=hover,
        hovertemplate='%{text}<extra></extra>',
        **common_args
    )

# ============================================
# FIGURE SETUP
# ============================================
initial_traces = []

for k in CLUSTER_OPTIONS:
    vis = (k == DEFAULT_CLUSTERS)

    t_total = make_trace(
        data_store[k][0]['Total'], f'Total Injuries ({k})', 'Blues',
        max_val_map[k], visible=vis
    )
    t_severe = make_trace(
        data_store[k][0]['Severe'], f'Severe Injuries ({k})', 'Reds',
        max_val_map[k], visible=vis
    )

    initial_traces.extend([t_total, t_severe])

fig = go.Figure(data=initial_traces)

# ============================================
# FRAMES
# ============================================
frames = []
for hour in range(24):
    frame_data = []
    for k in CLUSTER_OPTIONS:
        f_total = make_trace(data_store[k][hour]['Total'], f'Total Injuries ({k})', 'Blues', max_val_map[k], visible=None)
        f_severe = make_trace(data_store[k][hour]['Severe'], f'Severe Injuries ({k})', 'Reds', max_val_map[k], visible=None)
        frame_data.extend([f_total, f_severe])

    frames.append(go.Frame(
        data=frame_data,
        name=f'hour{hour}',
        layout=go.Layout(title_text=f"<b>Hour: {hour:02d}:00</b>")
    ))

fig.frames = frames

# ============================================
# DROPDOWN & LAYOUT
# ============================================
buttons = []
num_traces_per_k = 2

for i, k in enumerate(CLUSTER_OPTIONS):
    visibility = [False] * (len(CLUSTER_OPTIONS) * num_traces_per_k)
    visibility[2*i] = True
    visibility[2*i + 1] = True

    buttons.append(dict(
        label=f'{k} Clusters',
        method='update',
        args=[{'visible': visibility}, {}]
    ))

fig.update_layout(
    mapbox=dict(
        style='carto-positron',
        center=dict(lat=center_lat, lon=center_lon),
        zoom=10.5
    ),
    title=dict(text="<b>Severe vs Total Injury Hotspots</b>", y=0.96, x=0.5),
    height=800,
    margin=dict(b=100, t=80, r=50, l=50),
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.8)'),
    updatemenus=[dict(
        type='dropdown', direction='down', x=0.02, y=1.12, showactive=True, buttons=buttons
    )] + [dict(
        type='buttons', showactive=False, x=0.1, y=0.05,
        buttons=[
            dict(label='▶ Play', method='animate', args=[None, dict(frame=dict(duration=500, redraw=True), fromcurrent=True)]),
            dict(label='⏸ Pause', method='animate', args=[[None], dict(frame=dict(duration=0, redraw=False))])
        ]
    )],
    sliders=[dict(
        steps=[dict(method='animate', args=[[f'hour{k}'], dict(mode='immediate')], label=f'{k:02d}') for k in range(24)],
        active=0, y=0, x=0.1, len=0.8, pad=dict(t=20),
        currentvalue=dict(prefix="Hour: ", visible=True, xanchor='right')
    )]
)

fig.write_html(OUTPUT_FILE)
print(f"✅ Severity Map Generated (Darker): {OUTPUT_FILE}")

Total Crashes: 813,933
Crashes with Any Injury: 119,564
Crashes with Severe Injury: 14,189
Generating Severity Hotspots... [3, 5, 10, 20, 30]
✅ Severity Map Generated (Darker): severity_hotspots.html


-weekend vs weekday

-severe vs total injuries

-html files for 3,5,10,20 ... clusters